# SparkClient: Handling Batch Job Failures and Log Retrieval

This notebook demonstrates how to submit a Spark job expected to fail, wait for a failure status, retrieve crash logs for diagnosis, and clean up resources.

### What you will learn:
1. Submitting a remote `FileJob` with invalid parameters to trigger intentional application failure.
2. Waiting for a job to reach `SparkJobStatus.FAILED`.
3. Inspecting the failed job status and driver pod metadata.
4. Streaming driver logs to diagnose the failure reason.
5. Deleting the failed Spark job.

## 1. Imports and Client Initialization

Import necessary classes and initialize `SparkClient` with the target namespace.

In [ ]:
import os

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import FileJob, SparkClient, SparkJobStatus

namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
backend_config = KubernetesBackendConfig(namespace=namespace)
client = SparkClient(backend_config=backend_config)

print(f"SparkClient initialized for namespace: {namespace}")

## 2. Submit Job Expected to Fail

Submit Apache Spark's `pi.py` example passing `"hello"` as an argument instead of an integer. This causes the application to exit with an error.

In [ ]:
REMOTE_PI = "https://raw.githubusercontent.com/apache/spark/master/examples/src/main/python/pi.py"

print("Submitting Spark job expected to fail...")
job_name = client.submit_job(
    job=FileJob(
        file_source=REMOTE_PI,
        args=["hello"],
    ),
    num_executors=1,
    resources_per_executor={
        "cpu": "1",
        "memory": "512Mi",
    },
)

print(f"Job submitted successfully: {job_name}")

## 3. Wait for Failure Status

Monitor the application and wait until it transitions to `SparkJobStatus.FAILED`.

In [ ]:
print(f"Waiting for {job_name} to fail...")
job = client.wait_for_job_status(
    job_name,
    status={SparkJobStatus.FAILED},
    timeout=300,
)

print("Job failed as expected.")
print(f"Status: {job.status}")
print(f"Driver Pod: {job.driver_pod_name}")
print(f"Namespace: {job.namespace}")

## 4. Get Job Details

Retrieve job metadata using `get_job` to confirm application failure state.

In [ ]:
job = client.get_job(job_name)

print("Job details:")
print(f"Name: {job.name}")
print(f"Namespace: {job.namespace}")
print(f"Status: {job.status}")
print(f"Driver Pod: {job.driver_pod_name}")
print(f"Executors: {job.num_executors}")

## 5. Retrieve Driver Crash Logs

Stream the first 20 lines of driver logs to inspect the traceback and diagnose the failure.

In [ ]:
print(f"Retrieving logs for: {job_name}")
print("-" * 70)

line_count = 0
for line in client.get_job_logs(job_name):
    print(line.rstrip())
    line_count += 1
    if line_count >= 20:
        print("...")
        break

print("-" * 70)
print(f"Displayed {line_count} log lines.")

## 6. Clean Up Resources

Delete the failed Spark job from the cluster.

In [ ]:
print(f"Deleting job: {job_name}")
client.delete_job(job_name)
print("Job deleted successfully.")